In [5]:
import pandas as pd
import random
from copy import deepcopy

In [17]:
df = pd.read_csv('main.tsv', sep='\t')

In [18]:
models_list = ['SD3', 'SDXL', 'SDXL_Turbo', 'Kandinsky', 'PixArt_Sigma', 'Playground', 'IF', 'Openjourney', 'Hunyuan-DiT', 'SD_V1.5']

In [19]:
def sample_model_pair(models_list):
    return random.sample(models_list, 2)

# Adding the sampled pairs to the dataframe
df[['model_1', 'model_2']] = df.apply(lambda row: pd.Series(sample_model_pair(models_list)), axis=1)


In [20]:
df['model_1'].value_counts() + df['model_2'].value_counts()

Hunyuan-DiT     680
IF              663
Kandinsky       675
Openjourney     689
PixArt_Sigma    702
Playground      658
SD3             685
SDXL            686
SDXL_Turbo      651
SD_V1.5         651
dtype: int64

In [21]:
weights = [0.4, 0.4, 0.1, 0.1]  # Weights for values 0, 1, 2, 3, 4
values = [0, 1, 2, 3]

df['assessor_vitya'] = random.choices(values, weights=weights, k=len(df))

In [22]:
df['assessor_vitya'].value_counts()

1    1370
0    1348
3     331
2     321
Name: assessor_vitya, dtype: int64

In [59]:
import pandas as pd
import random
import argparse
import numpy as np
from statsmodels.stats.sandwich_covariance import cov_hc0

def expected_score(rating_a, rating_b):
    return 1 / (1 + 10**((rating_b - rating_a) / 400))

def update_elo(rating_a, rating_b, score_a, k=32):
    expected_a = expected_score(rating_a, rating_b)
    new_rating_a = rating_a + k * (score_a - expected_a)
    new_rating_b = rating_b + k * ((1 - score_a) - (1 - expected_a))
    return new_rating_a, new_rating_b

elo_ratings = {model: initial_elo for model in set(df['model_1']).union(df['model_2'])}
label_column='assessor_vitya'
output_file = 'ELO_test.txt'
ratings_changes = {model: [] for model in elo_ratings.keys()}

for index, row in df.iterrows():
    model_1 = row['model_1']
    model_2 = row['model_2']
    outcome = row[label_column]

    if outcome == 0:  # model_1 wins
        score_1 = 1
        score_2 = 0
    elif outcome == 1:  # model_2 wins
        score_1 = 0
        score_2 = 1
    elif outcome == 2:  # tie
        score_1 = 0.5
        score_2 = 0.5
    elif outcome == 3:  # both models are bad
        continue  # No change in ratings for this outcome

    rating_1 = elo_ratings[model_1]
    rating_2 = elo_ratings[model_2]

    new_rating_1, new_rating_2 = update_elo(rating_1, rating_2, score_1)

    ratings_changes[model_1].append(new_rating_1 - rating_1)
    ratings_changes[model_2].append(new_rating_2 - rating_2)

    elo_ratings[model_1] = new_rating_1
    elo_ratings[model_2] = new_rating_2

# Calculate standard deviation of rating changes
ratings_std = {model: np.std(changes) for model, changes in ratings_changes.items()}

# Calculate confidence intervals
z_score = 1.96  # For 95% confidence interval
confidence_intervals = {
    model: (elo_ratings[model] - z_score * ratings_std[model], elo_ratings[model] + z_score * ratings_std[model])
    for model in elo_ratings
}


# with open(output_file, 'w') as f:
#     for model, rating in elo_ratings.items():
#         ci = confidence_intervals[model]
#         f.write(f"{model}: {rating} (95% CI: {ci[0]:.2f} - {ci[1]:.2f})\n")




In [74]:
# https://github.com/bjlkeng/Bradley-Terry-Model/blob/master/update_model.py

import numpy as np
import os
import pandas as pd
import time

from datetime import datetime
from collections import Counter

DUMMY_PLAYER = 'DUMMY PLAYER'

def extract_game_data(df):
    df1 = deepcopy(df[df['assessor_vitya'] < 2])
    df1['Player A'] = df1['model_1']
    df1['Player B'] = df1['model_2']
    df1['Wins A'] = df1['assessor_vitya'] == 0
    df1['Wins B'] = df1['assessor_vitya'] == 1

    # assert all(c in df.columns for c in ['Date', 'Player A', 'Player B', 'Wins A', 'Wins B']), \
    #     'Expecting columns Date, Player A, Player B, Wins A, Wins B'

   # df['Date'] = df['Date'].astype(datetime)
    df1['Wins A'] = df1['Wins A'].astype(int)
    df1['Wins B'] = df1['Wins B'].astype(int)

    df1 = df1.drop(columns = df.columns)

    return df1



def add_dummy_games(game_data, alpha=1):
    ''' Regularizes the estimate by adding games against a dummy player.

        :param alpha: regularization parameter, number dummy wins/loses to add
    '''
    players = sorted(list(set(game_data['Player A']) | set(game_data['Player B'])))

    # Add dummy games
    dummy_data = [[p, DUMMY_PLAYER, alpha, alpha] for p in players]
    df = pd.DataFrame(dummy_data, columns=game_data.columns)
    df = pd.concat([game_data, df])
    df

    return df


def compute_rank_scores(game_data, max_iters=1000, error_tol=1e-3):
    ''' Computes Bradley-Terry using iterative algorithm

        See: https://en.wikipedia.org/wiki/Bradley%E2%80%93Terry_model
    '''
    # Do some aggregations for convenience
    # Total wins per player
    winsA = game_data.groupby('Player A').agg(sum)['Wins A'].reset_index()
    winsA = winsA[winsA['Wins A'] > 0]
    winsA.columns = ['Player', 'Wins']
    winsB = game_data.groupby('Player B').agg(sum)['Wins B'].reset_index()
    winsB = winsB[winsB['Wins B'] > 0]
    winsB.columns = ['Player', 'Wins']
    wins = pd.concat([winsA, winsB]).groupby('Player').agg(sum)['Wins']

    # Total games played between pairs
    num_games = Counter()
    for index, row in game_data.iterrows():
        key = tuple(sorted([row['Player A'], row['Player B']]))
        total = sum([row['Wins A'], row['Wins B']])
        num_games[key] += total

    # Iteratively update 'ranks' scores
    players = sorted(list(set(game_data['Player A']) | set(game_data['Player B'])))
    ranks = pd.Series(np.ones(len(players)) / len(players), index=players)
    for iters in range(max_iters):
        oldranks = ranks.copy()
        for player in ranks.index:
            denom = np.sum(num_games[tuple(sorted([player, p]))]
                           / (ranks[p] + ranks[player])
                           for p in ranks.index if p != player)
            ranks[player] = 1.0 * wins[player] / denom
        
        ranks /= sum(ranks)

        print(mle(ranks))
        if np.sum((ranks - oldranks).abs()) < error_tol:
            break

    if np.sum((ranks - oldranks).abs()) < error_tol:
        print(f" * Converged after {iters} iterations.")
    else:
        print(f" * Max iterations reached ({max_iters} iters).")

    del ranks[DUMMY_PLAYER]

    # Scale logarithm of score to be between 1 and 1000
    ranks_scaled = ranks.sort_values(ascending=False) \
                 .apply(lambda x: np.log1p(1000 * x) / np.log1p(1000) * 1000) \
                 .astype(int) \
                 .clip(1)

    return ranks_scaled, ranks

def mle(p):
    global games
    game_data = deepcopy(games)
    game_data = game_data[game_data['Player A'] != 'DUMMY PLAYER']    
    game_data = game_data[game_data['Player B'] != 'DUMMY PLAYER']

    winsA = game_data.groupby('Player A').agg(sum)['Wins A'].reset_index()
    winsA = winsA[winsA['Wins A'] > 0]
    winsA.columns = ['Player', 'Wins']
    winsB = game_data.groupby('Player B').agg(sum)['Wins B'].reset_index()
    winsB = winsB[winsB['Wins B'] > 0]
    winsB.columns = ['Player', 'Wins']
    wins = pd.concat([winsA, winsB]).groupby('Player').agg(sum)['Wins']

    pairwise = game_data.groupby(['Player A', 'Player B']).sum()

    total_models = len(p)
    players = sorted(list(set(game_data['Player A']) | set(game_data['Player B'])))
    ranks = pd.Series(np.ones(len(players)) / len(players), index=players)
    # for i in range(p):
    #     for j in range(p):
    #print(pairwise)
    l = 0
    for player_a in ranks.index:
        for player_b in ranks.index:
            if player_a == player_b:
                continue
            else:
                w_ij = pairwise.loc[player_a, player_b]['Wins A']
                l += (w_ij * np.log(p[player_a]) - w_ij * np.log(p[player_a] + p[player_b]))

    return l


#df = pd.read_csv('main.tsv', sep='\t')

df1 = extract_game_data(df)
games = add_dummy_games(df1)
ranks, raw_ranks = compute_rank_scores(games, error_tol=1e-5)

# BT_txt_path = "BT_ratings.txt"
# with open(BT_txt_path, 'w') as f:
#     for model, rating in ranks.items():
#         f.write(f"{model}: {rating}\n")

/tmp/ipykernel_493010/3246935825.py:56: FutureWarning: The operation <built-in function sum> failed on a column. If any error is raised, this will raise an exception in a future version of pandas. Drop these columns to avoid this warning.
  winsA = game_data.groupby('Player A').agg(sum)['Wins A'].reset_index()
/tmp/ipykernel_493010/3246935825.py:59: FutureWarning: The operation <built-in function sum> failed on a column. If any error is raised, this will raise an exception in a future version of pandas. Drop these columns to avoid this warning.
  winsB = game_data.groupby('Player B').agg(sum)['Wins B'].reset_index()
/tmp/ipykernel_493010/3246935825.py:77: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  denom = np.sum(num_games[tuple(sorted([player, p]))]
/tmp/ipykernel_493010/3246935825.py:109: FutureWarning: The operation <built-in function sum> failed o

-932.4436594255028
-932.085424622171
-932.0165810085049
-931.9985239073274
-931.99023105658
-931.9851578926484
-931.9819915692043
-931.9801020190124
-931.9790242453118
-931.9784310050067
-931.9781131600857
-931.9779463351231
 * Converged after 11 iterations.


/tmp/ipykernel_493010/3246935825.py:77: DeprecationWarning: Calling np.sum(generator) is deprecated, and in the future will give a different result. Use np.sum(np.fromiter(generator)) or the python sum builtin instead.
  denom = np.sum(num_games[tuple(sorted([player, p]))]
/tmp/ipykernel_493010/3246935825.py:109: FutureWarning: The operation <built-in function sum> failed on a column. If any error is raised, this will raise an exception in a future version of pandas. Drop these columns to avoid this warning.
  winsA = game_data.groupby('Player A').agg(sum)['Wins A'].reset_index()
/tmp/ipykernel_493010/3246935825.py:112: FutureWarning: The operation <built-in function sum> failed on a column. If any error is raised, this will raise an exception in a future version of pandas. Drop these columns to avoid this warning.
  winsB = game_data.groupby('Player B').agg(sum)['Wins B'].reset_index()
/tmp/ipykernel_493010/3246935825.py:77: DeprecationWarning: Calling np.sum(generator) is deprecated,

In [73]:
raw_ranks

Hunyuan-DiT     0.102270
IF              0.079600
Kandinsky       0.088058
Openjourney     0.098192
PixArt_Sigma    0.084735
Playground      0.078723
SD3             0.089905
SDXL            0.096552
SDXL_Turbo      0.098061
SD_V1.5         0.093241
dtype: float64

In [101]:
import torch
from functools import partial

def mle(inps,raw_ranks):
    global games
    game_data = deepcopy(games)
    game_data = game_data[game_data['Player A'] != 'DUMMY PLAYER']    
    game_data = game_data[game_data['Player B'] != 'DUMMY PLAYER']

    winsA = game_data.groupby('Player A').agg(sum)['Wins A'].reset_index()
    winsA = winsA[winsA['Wins A'] > 0]
    winsA.columns = ['Player', 'Wins']
    winsB = game_data.groupby('Player B').agg(sum)['Wins B'].reset_index()
    winsB = winsB[winsB['Wins B'] > 0]
    winsB.columns = ['Player', 'Wins']
    wins = pd.concat([winsA, winsB]).groupby('Player').agg(sum)['Wins']

    pairwise = game_data.groupby(['Player A', 'Player B']).sum()

    players = sorted(list(set(game_data['Player A']) | set(game_data['Player B'])))
    ranks = pd.Series(np.ones(len(players)) / len(players), index=players)
    # for i in range(p):
    #     for j in range(p):
    #print(pairwise)
    l = 0
    for player_a in ranks.index:
        for player_b in ranks.index:
            if player_a == player_b:
                continue
            else:
                idx_a = raw_ranks.index.get_loc(player_a)
                idx_b = raw_ranks.index.get_loc(player_b)
                w_ij = torch.tensor(pairwise.loc[player_a, player_b]['Wins A'])
                l += (w_ij * torch.log(inps[idx_a]) - w_ij * torch.log(inps[idx_a] + inps[idx_b]))

    return l

In [99]:
rank_tens = torch.tensor(raw_ranks.values)

In [103]:
hess = torch.autograd.functional.hessian(func=partial(mle, raw_ranks=raw_ranks), inputs=rank_tens)

/tmp/ipykernel_493010/1802199575.py:10: FutureWarning: The operation <built-in function sum> failed on a column. If any error is raised, this will raise an exception in a future version of pandas. Drop these columns to avoid this warning.
  winsA = game_data.groupby('Player A').agg(sum)['Wins A'].reset_index()
/tmp/ipykernel_493010/1802199575.py:13: FutureWarning: The operation <built-in function sum> failed on a column. If any error is raised, this will raise an exception in a future version of pandas. Drop these columns to avoid this warning.
  winsB = game_data.groupby('Player B').agg(sum)['Wins B'].reset_index()


In [107]:
hess

tensor([[ -6594.1178,   1027.8775,    772.5485,    745.5030,    800.2811,
            884.8583,    622.1028,    706.9579,    820.7278,    574.9637],
        [  1027.8775, -10308.2911,    712.7716,    759.7473,    890.3496,
           1239.1714,   1533.0903,    612.3668,    728.7671,   1172.9310],
        [   772.5485,    712.7716,  -8138.9297,    951.4238,   1240.7530,
           1151.9368,    884.5631,    938.5393,    981.1233,    882.7934],
        [   745.5030,    759.7473,    951.4238,  -6205.1953,   1105.9158,
           1022.6214,    960.5384,    737.3076,    803.8286,   1009.2428],
        [   800.2811,    890.3496,   1240.7530,   1105.9158, -10021.0991,
           1274.3100,   1049.8339,   1125.3833,    897.4974,    979.3060],
        [   884.8583,   1239.1714,   1151.9368,   1022.6214,   1274.3100,
         -11484.4026,    914.9506,   1106.3208,   1023.5752,    744.4801],
        [   622.1028,   1533.0903,    884.5631,    960.5384,   1049.8339,
            914.9506,  -8965.948

In [108]:
rank_tens = torch.tensor(raw_ranks.values, requires_grad=True)
out_mle = mle(rank_tens, raw_ranks=raw_ranks)

/tmp/ipykernel_493010/1802199575.py:10: FutureWarning: The operation <built-in function sum> failed on a column. If any error is raised, this will raise an exception in a future version of pandas. Drop these columns to avoid this warning.
  winsA = game_data.groupby('Player A').agg(sum)['Wins A'].reset_index()
/tmp/ipykernel_493010/1802199575.py:13: FutureWarning: The operation <built-in function sum> failed on a column. If any error is raised, this will raise an exception in a future version of pandas. Drop these columns to avoid this warning.
  winsB = game_data.groupby('Player B').agg(sum)['Wins B'].reset_index()


In [110]:
out_mle.backward()

In [112]:
rank_tens.grad

tensor([  56.3118,   26.1680,  -57.4529, -115.6094,   -4.7516,   45.5122,
          34.7352,   29.5661,   57.8144,  -67.1920], dtype=torch.float64)

In [117]:
B = rank_tens.grad.unsqueeze(0).T @ rank_tens.grad.unsqueeze(0)

In [121]:
V = (-hess).inverse() @ B @ (-hess).inverse()

In [123]:
V.numpy()

array([[0.01047975, 0.00813873, 0.00900934, 0.01005604, 0.00866868,
        0.0080529 , 0.00920402, 0.00989338, 0.0100476 , 0.00954498],
       [0.00813873, 0.00632066, 0.00699679, 0.00780967, 0.00673223,
        0.006254  , 0.00714798, 0.00768334, 0.00780311, 0.00741277],
       [0.00900934, 0.00699679, 0.00774524, 0.00864508, 0.00745238,
        0.006923  , 0.00791261, 0.00850524, 0.00863782, 0.00820572],
       [0.01005604, 0.00780967, 0.00864508, 0.00964946, 0.00831819,
        0.00772731, 0.00883189, 0.00949337, 0.00964136, 0.00915906],
       [0.00866868, 0.00673223, 0.00745238, 0.00831819, 0.00717059,
        0.00666123, 0.00761342, 0.00818364, 0.00831121, 0.00789545],
       [0.0080529 , 0.006254  , 0.006923  , 0.00772731, 0.00666123,
        0.00618805, 0.0070726 , 0.00760232, 0.00772083, 0.0073346 ],
       [0.00920402, 0.00714798, 0.00791261, 0.00883189, 0.00761342,
        0.0070726 , 0.00808359, 0.00868903, 0.00882447, 0.00838304],
       [0.00989338, 0.00768334, 0.0085052

In [124]:
from scipy.linalg import sqrtm

V_inv_sqrt = sqrtm(np.linalg.inv(V.numpy()))

In [129]:
from scipy.stats import chi2

alpha = 0.05  # your significance level (e.g., 0.05 for 95% confidence interval)
M = len(V_inv_sqrt)  # assuming V is square and dimension is the number of parameters
chi_squared_critical = chi2.ppf(1 - alpha, M - 1)

In [136]:
xi_hat = rank_tens.detach().numpy()  # your MLE estimate vector
T = 500  # or another value if defined in your context
chi_squared_critical_scaled = chi_squared_critical / T

# Initialize list to store confidence intervals
confidence_intervals = []

for i in range(M):
    unit_vector = np.zeros(M)
    unit_vector[i] = 1
    
    # Transform the unit vector by V_inv_sqrt
    transformed_vector = np.dot(V_inv_sqrt, unit_vector)
    
    # Calculate the margin of error
    margin_error = np.sqrt(chi_squared_critical) * np.linalg.norm(transformed_vector) / T
    
    # Compute the confidence interval for the ith parameter
    ci_lower = xi_hat[i] - margin_error
    ci_upper = xi_hat[i] + margin_error
    confidence_intervals.append((ci_lower, ci_upper))

# Output the confidence intervals for each parameter
for i, (ci_lower, ci_upper) in enumerate(confidence_intervals):
    print(f"Parameter {i}: Confidence Interval = ({ci_lower}, {ci_upper})")


Parameter 0: Confidence Interval = (-4392892.865874343, 4392893.070615668)
Parameter 1: Confidence Interval = (-3563043.351890175, 3563043.510895313)
Parameter 2: Confidence Interval = (-7205103.834292719, 7205104.010306854)
Parameter 3: Confidence Interval = (-3529816.9640021515, 3529817.160465444)
Parameter 4: Confidence Interval = (-3303471.0762742846, 3303471.24563299)
Parameter 5: Confidence Interval = (-5225254.149503893, 5225254.306832231)
Parameter 6: Confidence Interval = (-6739259.462354369, 6739259.64217191)
Parameter 7: Confidence Interval = (-6296592.614937545, 6296592.808222941)
Parameter 8: Confidence Interval = (-6237468.4572454365, 6237468.653543838)
Parameter 9: Confidence Interval = (-4759426.038140043, 4759426.224618777)


In [132]:
xi_hat = np.array(...)  # your MLE estimate vector
T = 1  # or another value if defined in your context




4.1132684819520895

In [134]:
from scipy.stats import norm

def scale(x):
    return np.log1p(1000 * x) / np.log1p(1000) * 1000

# Assuming V is your sandwich variance matrix and xi_hat is your MLE estimate vector
V = V  # your sandwich variance matrix
xi_hat =  rank_tens.detach().numpy()  # your MLE estimate vector
alpha = 0.05 # your significance level (e.g., 0.05 for 95% confidence interval)

# Extract diagonal elements (variances)
variances = np.diag(V)
standard_errors = np.sqrt(variances)

# Critical value for the standard normal distribution
z_critical = norm.ppf(1 - alpha / 2)

# Compute confidence intervals for each parameter
confidence_intervals = []
for i in range(len(xi_hat)):
    lower_bound = xi_hat[i] - z_critical * standard_errors[i]
    upper_bound = xi_hat[i] + z_critical * standard_errors[i]
    confidence_intervals.append((scale(lower_bound), scale(upper_bound)))

# Display the confidence intervals
for i, ci in enumerate(confidence_intervals):
    print(f"Confidence interval for parameter {i}: {ci}")

Confidence interval for parameter 0: (nan, 827.5112091351073)
Confidence interval for parameter 1: (nan, 791.0552700680457)
Confidence interval for parameter 2: (nan, 805.7060724691469)
Confidence interval for parameter 3: (nan, 821.5574361537633)
Confidence interval for parameter 4: (nan, 800.1486235528348)
Confidence interval for parameter 5: (nan, 789.5272847312503)
Confidence interval for parameter 6: (nan, 808.7887523909727)
Confidence interval for parameter 7: (nan, 819.2051425077144)
Confidence interval for parameter 8: (nan, 821.4363196574417)
Confidence interval for parameter 9: (nan, 814.0343862956362)


/tmp/ipykernel_493010/3402933676.py:4: RuntimeWarning: invalid value encountered in log1p
  return np.log1p(1000 * x) / np.log1p(1000) * 1000
